# Chinor Bank Assistant — Stage 0 baseline arms

Runs the three base-model arms on a T4 (fp16) and pushes `eval/outputs/<arm>/` to the branch `runs/c0-baselines`. No scoring happens here; scoring runs locally.

Secrets needed in Colab (key icon): `HF_TOKEN`, `GH_TOKEN` (a fine-grained token with *contents: write* on the repository).

In [ ]:
# Parameters
REPO = "Sarvarbek-Erniyazov/chinor-bank-assistant"
COMMIT = "main"            # commit SHA or branch to evaluate; the SHA is recorded in run.json
RUN_BRANCH = "runs/c0-baselines"
ARMS = [
    # (arm name, model id, shots)
    ("base-0.6B-0shot", "Qwen/Qwen3-0.6B-Base", 0),
    ("base-1.7B-0shot", "Qwen/Qwen3-1.7B-Base", 0),
    ("base-1.7B-3shot", "Qwen/Qwen3-1.7B-Base", 3),
]
BATCH_SIZE = 8

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
GH_TOKEN = userdata.get("GH_TOKEN")
assert os.environ["HF_TOKEN"] and GH_TOKEN

In [ ]:
import subprocess
if not os.path.isdir("repo"):
    subprocess.run(["git", "clone", f"https://{GH_TOKEN}@github.com/{REPO}.git", "repo"], check=True)
%cd repo
!git fetch --all --tags -q
!git checkout -q {COMMIT}
!git log --oneline -1

In [ ]:
# torch is not pinned: Colab's preinstalled torch is kept as is
!pip install -q -r requirements-gpu.txt

In [ ]:
# Colab's torchvision/torchaudio are built against its own torch; drop them so a
# version mismatch cannot break transformers ("operator torchvision::nms does not exist")
!pip uninstall -y -q torchvision torchaudio
import torch, transformers
try:
    from transformers import Qwen3ForCausalLM
except Exception as e:
    raise RuntimeError(f"transformers cannot import Qwen3ForCausalLM (torch {torch.__version__}); fix the environment before running the arms") from e
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

In [ ]:
for arm, model, shots in ARMS:
    !python eval/run_eval.py --model {model} --arm {arm} --shots {shots} --device cuda --dtype fp16 --batch-size {BATCH_SIZE}
    !tail -c 600 eval/outputs/{arm}/run.json

In [ ]:
!git config user.name "Sarvarbek Erniyazov"
!git config user.email "<email>"
!git checkout -q -B {RUN_BRANCH}
!git add -f eval/outputs/
!git commit -q -m "C0 baseline outputs: base-0.6B-0shot, base-1.7B-0shot, base-1.7B-3shot (T4 fp16)"
!git push -f https://{GH_TOKEN}@github.com/{REPO}.git {RUN_BRANCH}
!git log --oneline -1